In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,timestamp,server_id,region,server_type,cpu_usage,memory_usage,disk_io,network_latency,error_rate,temperature,health_score,alert_title,system_log,failure,anomaly_score
0,757,2025-03-01 00:00:00,srv_000,us-east,gpu-v100,56.28,48.81,73.10,0.92,0.40,56.05,61.77,Slow disk operations on srv_000,[00:03:03] WARN service.health: Check latency...,0.0,6
1,3475,2025-03-01 00:09:00,srv_000,eu-west,cpu-xeon,36.08,44.14,669.85,0.00,4.87,49.22,69.71,High memory consumption detected — srv_000,[00:06:32] INFO service.health: All checks pa...,NaN,20
2,4030,2025-03-01 01:38:00,srv_000,ap-east,cpu-xeon,100.00,67.40,0.00,41.51,0.00,63.88,35.31,Application errors spiking on srv_000,[01:04:37] INFO service.health: All checks pa...,1.0,93
3,24,2025-03-01 05:05:00,srv_000,eu-central,cpu-epyc,69.67,42.54,430.31,21.04,1.18,48.02,46.70,srv_000 approaching memory limit,[05:06:21] CRIT service: Cascading failure de...,NaN,47
4,3434,2025-03-01 05:48:00,srv_000,us-west,storage-nvme,NaN,79.48,748.55,150.85,0.81,44.48,46.98,Scheduled health check on srv_000,[05:03:21] INFO scheduler: Job j-7866 complet...,NaN,78


In [2]:
df_train.columns

Index(['id', 'timestamp', 'server_id', 'region', 'server_type', 'cpu_usage',
       'memory_usage', 'disk_io', 'network_latency', 'error_rate',
       'temperature', 'health_score', 'alert_title', 'system_log', 'failure',
       'anomaly_score'],
      dtype='object')

In [3]:
cat_features = ['server_id', 'region', 'server_type']
text_features = ['alert_title', 'system_log']
na_cols = ['cpu_usage', 'memory_usage', 'disk_io', 'network_latency', 'error_rate', 'temperature']

features = ['server_id', 'region', 'server_type', 'cpu_usage', 'memory_usage', 'disk_io', 'network_latency', 'error_rate', 'temperature', 'health_score', 'alert_title', 'system_log']
target1 = 'failure'
target2 = 'anomaly_score'

failure_df = df_train[df_train.failure.isna() == False]

X1 = failure_df[features]
y1 = failure_df[target1]

X2 = df_train[features]
y2 = df_train[target2]

X_final = df_test[features]

X1.head()

,server_id,region,server_type,cpu_usage,memory_usage,disk_io,network_latency,error_rate,temperature,health_score,alert_title,system_log
0,srv_000,us-east,gpu-v100,56.28,48.81,73.10,0.92,0.40,56.05,61.77,Slow disk operations on srv_000,[00:03:03] WARN service.health: Check latency...
2,srv_000,ap-east,cpu-xeon,100.00,67.40,0.00,41.51,0.00,63.88,35.31,Application errors spiking on srv_000,[01:04:37] INFO service.health: All checks pa...
5,srv_000,eu-west,storage-hdd,4.58,42.89,811.85,92.55,0.57,51.81,27.42,srv_000: periodic status report,[06:02:55] INFO auth: Session refresh for use...
6,srv_000,us-east,storage-hdd,9.85,NaN,0.00,54.08,11.28,39.12,28.90,HIGH CPU LOAD ALER TFOR SRV_000,[08:04:10] CRIT kernel: Hardware ECC error on...
9,srv_000,ap-south,cpu-epyc,50.56,NaN,662.25,26.57,0.00,70.86,83.49,SYSTEM CHECK COMPLETED FOR SRV_000,[10:07:47] DEBUG storage: Block allocation nor...


In [4]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

model = CatBoostClassifier(cat_features=cat_features, text_features=text_features, auto_class_weights='Balanced', random_state=42, silent=True)
model.fit(X1, y1)
preds1 = model.predict(X_final)

In [5]:
df_1 = pd.DataFrame({
    "id": df_test.id,
    "subtaskID": 1,
    "answer": preds1
})

df_1.head()

,id,subtaskID,answer
0,0,1,0.0
1,1,1,0.0
2,2,1,0.0
3,3,1,0.0
4,4,1,0.0


In [6]:
df_1.answer.value_counts()

answer
0.0    1100
1.0     400
Name: count, dtype: int64

In [7]:
from catboost import CatBoostRegressor

model = CatBoostRegressor(cat_features=cat_features, text_features=text_features, random_state=42, silent=True)
model.fit(X2, y2)
preds2 = model.predict(X_final)

In [8]:
df_2 = pd.DataFrame({
    "id": df_test.id,
    "subtaskID": 2,
    "answer": preds2
})

df_2.head()

,id,subtaskID,answer
0,0,2,69.496404
1,1,2,60.704351
2,2,2,65.269821
3,3,2,60.239241
4,4,2,63.122730


In [9]:
submission = pd.concat([df_1, df_2])
submission

,id,subtaskID,answer
0,0,1,0.000000
1,1,1,0.000000
2,2,1,0.000000
3,3,1,0.000000
4,4,1,0.000000
...,...,...,...
1495,1495,2,77.650453
1496,1496,2,70.818888
1497,1497,2,67.155066
1498,1498,2,80.093316


In [10]:
submission.to_csv("submission.csv", index=False)